In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from collections import Counter

def pstats(x):
    print("mean: " + str(np.mean(x)))
    print("std: " + str(np.std(x)))
    print("min: " + str(np.min(x)))
    print("max: " + str(np.max(x)))
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


In [2]:
# Load data
with np.load('/home/server/Projects/data/AKI/preop_trainable/normalized.npz', allow_pickle=True) as data:
    X_train = data["X_train_normalized"]
    X_test = data["X_test_normalized"]
    y_train = data["y_train_normalized"]  # Continuous target (for regression models)
    y_test = data["y_test_normalized"]
    y_binary_train = data["y_binary_train_normalized"]
    y_binary_test = data["y_binary_test_normalized"]


In [3]:
# -------------------- Base Models --------------------
# Logistic Regression (Used Directly)
logistic = LogisticRegression(max_iter=1000)

# Linear Regression (Will Convert Predictions Using `> 0.3`)
linear = LinearRegression()


mlp = MLPClassifier()

# K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=250, weights='distance')

# Support Vector Machine (SVM)
# svm = SVC(kernel='rbf', probability=True, random_state=42) #https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html#sklearn.svm.LinearSVC
# ^fit time scales quadratically with the number of samples which makes it hard to scale to datasets with more than a couple 10000 samples
svm = LinearSVC(random_state=42)

# Random Forest
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)

# XGBoost
xgb_classifier = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [4]:
# -------------------- Train Regression Models Separately --------------------
# Train Linear Regression
linear.fit(X_train, y_train)

# Predict probabilities using Linear Regression & threshold at 0.3
y_pred_linear = (linear.predict(X_test) > 0.3).astype(int)

In [ ]:
# -------------------- Ensemble Model --------------------
# Custom Class to add predict_proba method to LinearSVC
def ExpandedLinearSVC(LinearSVC):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
    def predict_proba(self, X):
        df = self.decision_function(X)
        return np.array([sigmoid(x) for x in df])

In [ ]:
# -------------------- Ensemble Model --------------------
# Custom Class for Including Linear Regression in VotingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin



class LinearRegressionBinaryClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, threshold=0.3):
        self.threshold = threshold
        self.model = LinearRegression()

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict(self, X):
        y_pred = self.model.predict(X)
        return (y_pred > self.threshold).astype(int)
    
    def predict_proba(self, X):
        y_pred = self.model.predict(X)
        return sigmoid(y_pred - self.threshold)

In [6]:
# Wrapping Linear Regression for VotingClassifier
linear_binary = LinearRegressionBinaryClassifier(threshold=0.3)


In [ ]:
model = LinearSVC(random_state=42)
model.fit(X_train, y_binary_train)

In [18]:
y_pred = model.predict(X_test)
print(Counter(y_pred))
print(Counter(y_binary_test))
pstats(model.decision_function(X_test))

Counter({False: 7355, True: 2900})
Counter({False: 9596, True: 659})
mean: -0.18221956140050066
std: 0.4131180141137641
min: -1.8123342155839133
max: 2.2923842479377834


In [19]:
pstats(y_test)
    

mean: 0.030145294978059487
std: 0.29646887954605994
min: -1.71
max: 4.63


In [ ]:
logistic = LogisticRegression(max_iter=1000)
logistic.fit(X_train, y_binary_train)
y_pred = logistic.predict(X_train)

In [ ]:
# Wrapping Linear Regression for VotingClassifier
# MLP_binary = MLPClassifier(threshold=0.3)

In [25]:
# Voting Classifier (Soft Voting: Uses Predicted Probabilities)
ensemble = VotingClassifier(
    estimators=[
        ('logistic', logistic),   #works
        # ('linear', linear_binary),
        # ('knn', knn),             #works
        # ('svm', svm),
        # ('random_forest', random_forest),
        # ('xgb', xgb_classifier)
    ],
    voting='soft'  # Uses probability-based voting
)

In [26]:
# Train Ensemble Model
ensemble.fit(X_train, y_binary_train)


VotingClassifier(estimators=[('logistic', LogisticRegression(max_iter=1000))],
                 voting='soft')

In [27]:
# Predict binary labels
y_pred = ensemble.predict(X_test)

# Classification Metrics
accuracy = accuracy_score(y_binary_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

cm = confusion_matrix(y_binary_test, y_pred)
print('Confusion Matrix:')
print(cm)

report = classification_report(y_binary_test, y_pred)
print('Classification Report:')
print(report)

Accuracy: 0.73
Confusion Matrix:
[[7070 2526]
 [ 225  434]]
Classification Report:
              precision    recall  f1-score   support

       False       0.97      0.74      0.84      9596
        True       0.15      0.66      0.24       659

    accuracy                           0.73     10255
   macro avg       0.56      0.70      0.54     10255
weighted avg       0.92      0.73      0.80     10255



In [ ]:
# Predict probabilities for ROC curve
y_prob = ensemble.predict_proba(X_test)[:, 1]

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_binary_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot ROC Curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Full Ensemble Model: ROC Curve')
plt.legend(loc='lower right')
plt.show()